# 01 — Parcellate Individual Surface Contrast Maps

For each subject × task × contrast × encounter, load the hemi-L and hemi-R `.func.gii` effect-size maps
and compute the mean activation within each Schaefer 400 parcel.

**Output:** `processed_data/surface_parcel_indiv.pkl`
  — `parcel_dict[subject][task][contrast][enc_key]` = DataFrame(region, activation, network, roi_value)

In [1]:
import sys
import pickle
import numpy as np
import nibabel as nib
import pandas as pd
from pathlib import Path
from collections import defaultdict

try:
    _nb_dir = Path(__file__).resolve().parent
except NameError:
    _nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir))
from config import (
    TASKS, CONTRASTS, SUBJECTS, SESSIONS, RT_CONTRAST,
    N_PARCELS, DATA_DIR,
    build_surface_contrast_map_path,
    get_schaefer_surface_labels,
    parse_network,
)

/oak/stanford/groups/russpold/users/nklevak/network_second_modeling/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 3.2.1'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
labels, parcel_names = get_schaefer_surface_labels()
networks = [parse_network(n) for n in parcel_names]
roi_values = list(range(1, N_PARCELS + 1))

print(f'Labels shape: {labels.shape}')
print(f'Parcels: {N_PARCELS}, e.g. {parcel_names[:3]}')

Labels shape: (81924,)
Parcels: 400, e.g. ['Background', '7Networks_LH_Vis_1', '7Networks_LH_Vis_2']


In [3]:
def compute_parcel_means(vertex_data, labels, n_parcels=400):
    """Mean activation per parcel. vertex_data: (81924,), labels: (81924,) with 1-indexed parcels."""
    sums   = np.bincount(labels, weights=vertex_data.astype(float), minlength=402)
    counts = np.bincount(labels, minlength=402)
    with np.errstate(divide='ignore', invalid='ignore'):
        means = np.where(counts[1:n_parcels+1] > 0,
                         sums[1:n_parcels+1] / counts[1:n_parcels+1], 0.0)
    return means   # (400,)


def load_gifti_concat(lh_path, rh_path):
    """Load both hemisphere .func.gii files and return concatenated (81924,) array."""
    lh = nib.load(lh_path).darrays[0].data
    rh = nib.load(rh_path).darrays[0].data
    return np.concatenate([lh, rh])

In [4]:
parcel_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        for subject in SUBJECTS:
            enc_count = 0
            for session in SESSIONS:
                lh_path = build_surface_contrast_map_path(subject, session, task, contrast, 'L')
                rh_path = build_surface_contrast_map_path(subject, session, task, contrast, 'R')
                if lh_path.exists() and rh_path.exists():
                    enc_key = f'{enc_count + 1:02d}'
                    vertex_data = load_gifti_concat(lh_path, rh_path)
                    means = compute_parcel_means(vertex_data, labels)
                    parcel_dict[subject][task][contrast][enc_key] = pd.DataFrame({
                        'region':     parcel_names,
                        'activation': means,
                        'network':    networks,
                        'roi_value':  roi_values,
                    })
                    enc_count += 1

print('Done loading.')

ValueError: All arrays must be of the same length

In [ ]:
# Summary
for subject in SUBJECTS:
    for task in TASKS:
        for contrast in CONTRASTS[task]:
            if contrast == RT_CONTRAST:
                continue
            n = len(parcel_dict[subject][task].get(contrast, {}))
            status = f'{n} encounters' if n > 0 else 'MISSING'
            print(f'  {subject} | {task} | {contrast}: {status}')

In [ ]:
out_path = DATA_DIR / 'surface_parcel_indiv.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(dict(parcel_dict), f)
print(f'Saved to {out_path}')